# 📊 Master Audit Dashboard: TRAIN, VAL & TEST Splits
### Real-Time Tensor & Progress Inspector under `THESIS_MOTHERFILE/baseline_training/`

Sinusuri nito ang buong pipeline sa loob ng:
1. **`baseline_training/train/`** (14,815 clips across 6 datasets)
2. **`baseline_training/val/`** (1,457 clips: Tracks+MELD & MOSEI+Mustard)
3. **`baseline_training/test/`** (1,469 clips: Tracks+MELD & MOSEI+Mustard)

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
print('Google Drive mounted successfully!')

## Step 2: Audit TRAIN, VAL, and TEST Tensor Progress

In [ ]:
import json, glob, os
from pathlib import Path
import pandas as pd

BASE_DIR = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/baseline_training')
# Fallback if user has 'Baseline preprocessed'
if not BASE_DIR.exists():
    ALT_DIR = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline preprocessed')
    if ALT_DIR.exists():
        BASE_DIR = ALT_DIR

SPLIT_CONFIGS = {
    'TRAIN': {
        'dir': BASE_DIR if (BASE_DIR / 'CMU-MOSEI').exists() else (BASE_DIR / 'train'),
        'target': 14815,
        'datasets': ['CMU-MOSEI', 'MELD', 'TRACK_1', 'MUSTARD', 'TRACK_2', 'TRACK_3']
    },
    'VAL': {
        'dir': BASE_DIR / 'val',
        'target': 1457,
        'datasets': ['TRACKS_MELD', 'MOSEI_MUSTARD']
    },
    'TEST': {
        'dir': BASE_DIR / 'test',
        'target': 1469,
        'datasets': ['TRACKS_MELD', 'MOSEI_MUSTARD']
    }
}

print('=' * 85)
print('      MASTER AUDIT DASHBOARD: TRAIN, VAL & TEST PREPROCESSING')
print(f'      Target Path: {BASE_DIR}')
print('=' * 85)

summary_data = []

for split_name, cfg in SPLIT_CONFIGS.items():
    s_root = cfg['dir']
    s_target = cfg['target']
    d_list = cfg['datasets']
    
    tot_completed = 0
    tot_audios = 0
    tot_texts = 0
    tot_visuals = 0
    
    print(f'\n📂 === [{split_name} SET] (Target: {s_target:,} clips) ===')
    print('-' * 85)
    print(f"  {'Group / Dataset':<18} | {'Audio (.npy)':<14} | {'Text (.npy)':<14} | {'Visual Folders':<16} | {'Status'}")
    print('  ' + '-' * 81)
    
    for dname in d_list:
        d_path = s_root / dname
        aud_count = len(list(d_path.glob('**/*_melspec.npy')))
        txt_count = len(list(d_path.glob('**/*_input_ids.npy')))
        vis_count = len([x for x in d_path.glob('**/visual/*') if x.is_dir()])
        
        tot_audios += aud_count
        tot_texts += txt_count
        tot_visuals += vis_count
        
        status = '🟢 READY' if aud_count > 0 else '⚪ EMPTY'
        print(f"  {dname:<18} | {aud_count:>12,} | {txt_count:>12,} | {vis_count:>14,} | {status}")
        
    pct = (tot_audios / s_target) * 100 if s_target > 0 else 0
    print('  ' + '-' * 81)
    print(f"  👉 {split_name} Subtotal: {tot_audios:,} / {s_target:,} extracted tensors ({pct:.1f}%)")
    
    summary_data.append({
        'Split': split_name,
        'Target Clips': s_target,
        'Extracted Tensors': tot_audios,
        'Visual Folders': tot_visuals,
        'Completion (%)': f"{pct:.1f}%",
        'Ready For Training': 'YES ✅' if pct >= 90.0 else 'IN PROGRESS ⏳'
    })

print('\n' + '=' * 85)
print('                   🏆 FINAL PIPELINE READINESS SUMMARY 🏆')
print('=' * 85)
df = pd.DataFrame(summary_data)
print(df.to_string(index=False))
print('=' * 85)